### Data ingestion -> VectorDB

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [5]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: C11_2203084_Prateek Khemchandani.pdf
  ✓ Loaded 59 pages

Total documents loaded: 59


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-04-20T00:24:34+05:30', 'author': 'PRATEEKK', 'moddate': '2026-04-20T00:24:34+05:30', 'source': '..\\data\\pdf\\C11_2203084_Prateek Khemchandani.pdf', 'total_pages': 59, 'page': 0, 'page_label': '1', 'source_file': 'C11_2203084_Prateek Khemchandani.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-04-20T00:24:34+05:30', 'author': 'PRATEEKK', 'moddate': '2026-04-20T00:24:34+05:30', 'source': '..\\data\\pdf\\C11_2203084_Prateek Khemchandani.pdf', 'total_pages': 59, 'page': 1, 'page_label': '2', 'source_file': 'C11_2203084_Prateek Khemchandani.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-04-20T00:24:34+05:30', 'author': 'PRATEEKK', 'moddate': '2026-04-20T00:24:3

In [12]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [13]:
chunks=split_documents(all_pdf_documents)
chunks

Split 59 documents into 18 chunks

Example chunk:
Content: CODE: 
 
// Server.java 
import java.io.*; 
import java.net.*; 
 
public class Server { 
public static void main(String[] args) throws Exception { 
 
ServerSocket serverSocket = new ServerSocket(9999)...
Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-04-20T00:24:34+05:30', 'author': 'PRATEEKK', 'moddate': '2026-04-20T00:24:34+05:30', 'source': '..\\data\\pdf\\C11_2203084_Prateek Khemchandani.pdf', 'total_pages': 59, 'page': 6, 'page_label': '7', 'source_file': 'C11_2203084_Prateek Khemchandani.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-04-20T00:24:34+05:30', 'author': 'PRATEEKK', 'moddate': '2026-04-20T00:24:34+05:30', 'source': '..\\data\\pdf\\C11_2203084_Prateek Khemchandani.pdf', 'total_pages': 59, 'page': 6, 'page_label': '7', 'source_file': 'C11_2203084_Prateek Khemchandani.pdf', 'file_type': 'pdf'}, page_content='CODE: \n \n// Server.java \nimport java.io.*; \nimport java.net.*; \n \npublic class Server { \npublic static void main(String[] args) throws Exception { \n \nServerSocket serverSocket = new ServerSocket(9999); \nSystem.out.println("Server is running..."); \n \nSocket socket = serverSocket.accept(); \nSystem.out.println("Client connected"); \n \nBufferedReader in = new BufferedReader(new \nInputStreamReader(socket.getInputStream())); \nPrintWriter out = new PrintWriter(socket.getOutputStream(), true); \nString message; \nwhile (true) { \nmessage = in.readLine(); \n \nif (message == null || m